# Phase 2 — Candidate sites for the Azerbaijan wind project (GWA data)

**Output** (`/content/phase2_out`): `candidate_sites.csv`, `accessibility_points.csv`, `region_config.json`, PNG plots.

**Workflow:** run cells top to bottom. Cell **E** lists accessibility candidates and **stops on purpose** until you put your chosen IDs into `CONFIG["CHOSEN_IDS"]` (cell 2) — then re-run from cell **E2**.

Upload to Colab beforehand: the three `*wind-speed*{50,100,150}m.tif` files into `/content/gwa/` and `custom.geojson` into `/content/`.

In [ ]:
# 1. Imports (Colab usually has these; install only if missing)
import importlib, subprocess, sys
for mod, pipname in [("rasterio","rasterio"),("geopandas","geopandas"),("shapely","shapely"),("pyproj","pyproj"),("scipy","scipy")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pipname])
print("imports OK")

In [ ]:
# 2. CONFIG  (all paths / switches live here)
CONFIG = dict(
    DATA_DIR="/content",                     # folder with the GWA tifs (searched recursively); tifs are directly in /content
    POLYGON_PATH="/content/custom.geojson",
    OUT_DIR="/content/phase2_out",
    HEIGHTS=[50, 100, 150],
    WS_PATHS={},                             # optional override, e.g. {50:"/x/a.tif", 100:"...", 150:"..."}
    CF_PATHS={},                             # optional 100 m capacity-factor tifs: {"iec1":..., "iec2":..., "iec3":...}
    CF_REQUIRED=False,                       # False -> CF columns are NaN when files are missing
    LAND_MODE="naturalearth",                # "naturalearth" | "polygon_only" (diagnostic only: NO land mask)
    NE_LAND_URL="https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip",    # verify URL if download fails
    NE_LAKES_URL="https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_lakes.zip",  # verify URL if download fails
    NE_LAND_PATH=None, NE_LAKES_PATH=None,   # manual fallback: local .zip/.shp/.geojson of ne_10m_land / ne_10m_lakes
    SHORE_BUFFER_M=1000,                     # sites must be >= this far inland / from lakes
    BBOX_PAD_DEG=0.002,                      # inward pad of the grid bbox (bilinear needs 4 neighbours)
    TARGET_N=250, N_MIN=200, N_MAX=300,
    SPACING_RANGE_KM=(1.0, 8.0), SPACING_STEP_KM=0.05,
    UTM="EPSG:32639",
    ACCESS_SOURCE="overpass",                # "overpass" | "csv"
    ACCESS_CSV="/content/access_manual.csv", # columns: id,type,lat,lon,note   (type = road | substation)
    OVERPASS_URLS=["https://overpass-api.de/api/interpreter",
                   "https://overpass.private.coffee/api/interpreter"],   # mirrors; check they are alive
    JCT_CLUSTER_KM=2.0,                      # merge road junctions closer than this
    CHOSEN_IDS=[],                           # <-- YOUR CHOICE, e.g. ["SUB01","SUB03","JCT02","JCT05","JCT09"]
    MIN_POINTS=5, MAX_POINTS=7, MIN_SUBSTATIONS=2,
    ACCESS_DECAY_PER_KM=0.5,                 # access_score = exp(-0.5 * d_km)
    SEED=42,
)

In [ ]:
# 3. Load rasters + polygon, run input checks (a)
import os, re, glob, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.features import geometry_mask
from shapely.geometry import box
from shapely.ops import transform as shp_transform, unary_union
from shapely import contains_xy
from pyproj import Transformer
from scipy.interpolate import RegularGridInterpolator
import matplotlib; import matplotlib.pyplot as plt

C = CONFIG; np.random.seed(C["SEED"]); OUT = Path(C["OUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)

def find_ws(h):
    if h in C["WS_PATHS"]: return C["WS_PATHS"][h]
    pat = re.compile(rf"wind[-_ ]?speed.*(?<!\d){h}\s*m", re.I)   # (?<!\d): 50m must not match 150m
    hits = [p for p in glob.glob(os.path.join(C["DATA_DIR"], "**", "*.tif*"), recursive=True) if pat.search(os.path.basename(p))]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 wind-speed {h} m tif in {C['DATA_DIR']}, found {hits}. Set CONFIG['WS_PATHS'].")
    return hits[0]

REF = {}
def read_layer(path):
    with rasterio.open(path) as r:
        arr = r.read(1).astype("float64")
        if r.nodata is not None and not np.isnan(r.nodata): arr[arr == r.nodata] = np.nan
        meta = dict(crs=r.crs.to_string() if r.crs else None, w=r.width, h=r.height, tr=tuple(r.transform)[:6])
    if not REF: REF.update(meta)
    else:
        same = meta["crs"] == REF["crs"] and meta["w"] == REF["w"] and meta["h"] == REF["h"] and np.allclose(meta["tr"], REF["tr"], atol=1e-9)
        if not same: raise ValueError(f"Grid/CRS mismatch: {path}\n  got {meta}\n  ref {REF}")
    return arr

WS_PATHS = {h: find_ws(h) for h in C["HEIGHTS"]}
WS = {h: read_layer(p) for h, p in WS_PATHS.items()}

# optional CF layers
CF_PATHS = dict(C["CF_PATHS"])
if not CF_PATHS:   # loose auto-detect (file naming NOT verified): needs 'capacity' and iec1/iec2/iec3 in the name
    for k in ("iec1", "iec2", "iec3"):
        hits = [p for p in glob.glob(os.path.join(C["DATA_DIR"], "**", "*.tif*"), recursive=True)
                if re.search("capacity", os.path.basename(p), re.I) and re.search(k.replace("iec", "iec[-_ ]?"), os.path.basename(p), re.I)]
        if len(hits) == 1: CF_PATHS[k] = hits[0]
        elif len(hits) > 1: raise ValueError(f"Several CF files match {k}: {hits}. Set CONFIG['CF_PATHS'].")
CF = {k: read_layer(p) for k, p in CF_PATHS.items()}
if C["CF_REQUIRED"] and set(CF) != {"iec1", "iec2", "iec3"}:
    raise FileNotFoundError(f"CF_REQUIRED=True but found only {sorted(CF)}")

poly_gdf = gpd.read_file(C["POLYGON_PATH"])
poly_gdf = poly_gdf.set_crs(4326) if poly_gdf.crs is None else poly_gdf.to_crs(4326)
if len(poly_gdf) != 1 or poly_gdf.geometry.iloc[0].geom_type != "Polygon": raise ValueError("Expected exactly one Polygon feature")
poly_ll = poly_gdf.geometry.iloc[0]

TR = rasterio.transform.Affine(*REF["tr"]); H, W = REF["h"], REF["w"]
if REF["crs"] != "EPSG:4326": raise ValueError(f"Rasters are {REF['crs']}, expected EPSG:4326")
rbox = box(TR.c, TR.f + TR.e * H, TR.c + TR.a * W, TR.f)
if not rbox.covers(poly_ll): raise ValueError("Polygon is not fully covered by the raster extent")

# ---- checks (a)
inside = geometry_mask([poly_ll], (H, W), TR, invert=True)
print(f"raster {W}x{H}, px {TR.a:.4f} deg, bounds lon {rbox.bounds[0]:.5f}-{rbox.bounds[2]:.5f} lat {rbox.bounds[1]:.5f}-{rbox.bounds[3]:.5f}")
for h, a in WS.items():
    v = a[np.isfinite(a)]
    print(f"ws{h}: mean {v.mean():.2f} ({v.min():.2f}-{v.max():.2f}) | NaN total {np.isnan(a).sum()} | NaN inside polygon {np.isnan(a[inside]).sum()}")
ok = inside & np.all([np.isfinite(WS[h]) for h in C["HEIGHTS"]], axis=0)
share = np.mean((WS[50][ok] < WS[100][ok]) & (WS[100][ok] < WS[150][ok]))
print(f"ws50<ws100<ws150 for {share*100:.2f}% of valid pixels inside polygon")
print("CF layers found:", sorted(CF) or "none (columns will be NaN)")

# ---- projections, interpolators
to_utm = Transformer.from_crs(4326, C["UTM"], always_xy=True); to_ll = Transformer.from_crs(C["UTM"], 4326, always_xy=True)
poly_utm = shp_transform(to_utm.transform, poly_ll)
print(f"polygon area: {poly_utm.area/1e6:.1f} km2 (UTM)")

LON_C = TR.c + (np.arange(W) + 0.5) * TR.a
LAT_ASC = (TR.f + (np.arange(H) + 0.5) * TR.e)[::-1]            # rows run north->south, flip for ascending axis
def make_interp(arr):
    return RegularGridInterpolator((LAT_ASC, LON_C), arr[::-1, :], method="linear", bounds_error=False, fill_value=np.nan)
INTERP = {f"ws{h}": make_interp(a) for h, a in WS.items()}
INTERP.update({f"cf100_{k}": make_interp(a) for k, a in CF.items()})
def sample(name, lon, lat): return INTERP[name](np.column_stack([lat, lon]))

# grid bbox: polygon bounds shrunk by pad
minx, miny, maxx, maxy = poly_ll.bounds; p = C["BBOX_PAD_DEG"]
BBOX_LL = (minx + p, miny + p, maxx - p, maxy - p)
cx, cy = to_utm.transform([BBOX_LL[0], BBOX_LL[2], BBOX_LL[0], BBOX_LL[2]], [BBOX_LL[1], BBOX_LL[1], BBOX_LL[3], BBOX_LL[3]])
X0, X1, Y0, Y1 = min(cx), max(cx), min(cy), max(cy)
REQ = [f"ws{h}" for h in C["HEIGHTS"]] + ([f"cf100_{k}" for k in ("iec1", "iec2", "iec3")] if C["CF_REQUIRED"] else [])

In [ ]:
# 4. Land mask (b): Natural Earth 10m land - lakes (Caspian), shrunk by SHORE_BUFFER_M, intersected with polygon
import requests
def _load_vec(url, path, name):
    cache = OUT / "_cache"; cache.mkdir(exist_ok=True)
    if path: return gpd.read_file(f"zip://{path}" if str(path).endswith(".zip") else path)
    try:
        f = cache / f"{name}.zip"
        if not f.exists():
            r = requests.get(url, timeout=90); r.raise_for_status(); f.write_bytes(r.content)
        return gpd.read_file(f"zip://{f}")
    except Exception as e:
        print(f"[!] download of {name} failed: {repr(e)[:150]}")
        try:   # Colab manual upload fallback
            from google.colab import files
            print(f"Upload {name}.zip (Natural Earth 10m {name}, from naturalearthdata.com -> Downloads -> 10m Physical Vectors):")
            up = files.upload(); fn = next(iter(up)); f.write_bytes(up[fn]); return gpd.read_file(f"zip://{f}")
        except ImportError:
            raise RuntimeError(f"Could not get {name}. Download it manually and set CONFIG['NE_{name.split('_')[-1].upper()}_PATH'].")

if C["LAND_MODE"] == "polygon_only":
    print("[!] LAND_MODE=polygon_only: NO land mask applied (diagnostic only; sea points will be kept!)")
    LAND_UTM = poly_utm; MASK_SOURCE = "none (polygon only, diagnostic)"; MASK_UTM = poly_utm
else:
    land = _load_vec(C["NE_LAND_URL"], C["NE_LAND_PATH"], "ne_10m_land")
    lakes = _load_vec(C["NE_LAKES_URL"], C["NE_LAKES_PATH"], "ne_10m_lakes")
    win = box(*poly_ll.bounds).buffer(0.2)                       # work only near the polygon
    lu = unary_union(land[land.intersects(win)].to_crs(C["UTM"]).geometry)
    ku = unary_union(lakes[lakes.intersects(win)].to_crs(C["UTM"]).geometry)
    LAND_UTM = lu.difference(ku)                                 # land without lakes / Caspian
    MASK_UTM = LAND_UTM.buffer(-C["SHORE_BUFFER_M"]).intersection(poly_utm)
    MASK_SOURCE = "Natural Earth 10m land minus 10m lakes, eroded by %d m, intersected with polygon" % C["SHORE_BUFFER_M"]
    print(f"polygon area {poly_utm.area/1e6:.0f} km2 | land inside polygon {LAND_UTM.intersection(poly_utm).area/1e6:.0f} km2 | "
          f"after {C['SHORE_BUFFER_M']} m buffer {MASK_UTM.area/1e6:.0f} km2")
    if MASK_UTM.is_empty: raise RuntimeError("Land mask is empty - check the Natural Earth files")
print("mask:", MASK_SOURCE)

**Limits of the land mask:** Natural Earth 10m is a 1:10 million generalized dataset (shore errors of hundreds of metres are normal; small bays, salt flats, Absheron lakes may be missing or wrong). The Caspian level changes (it has fallen in recent decades, so the real shore may be *further* than NE shows, and some "sea" pixels may now be land). The 1 km inward buffer absorbs part of this error but does not remove it. Whether NE's *land* layer already excludes the Caspian is not something I verified; the code subtracts the *lakes* layer anyway, so it works either way (see the printed land area).

In [ ]:
# 5. Grid + automatic spacing (c)
def classify(s_km, return_all=False):
    s = s_km * 1000
    xs = np.arange(X0 + s / 2, X1, s); ys = np.arange(Y0 + s / 2, Y1, s)
    if len(xs) == 0 or len(ys) == 0: return None
    XX, YY = np.meshgrid(xs, ys[::-1]); x, y = XX.ravel(), YY.ravel()   # north->south, west->east
    lon, lat = to_ll.transform(x, y)
    in_poly = contains_xy(poly_utm, x, y)
    on_land = contains_xy(MASK_UTM, x, y)
    nanmask = np.zeros(len(x), bool)
    for n in REQ: nanmask |= np.isnan(sample(n, lon, lat))
    status = np.where(~in_poly, "outside_polygon", np.where(~on_land, "not_land", np.where(nanmask, "nan", "kept")))
    return dict(x=x, y=y, lon=lon, lat=lat, status=status)

rows = []
for s in np.arange(C["SPACING_RANGE_KM"][0], C["SPACING_RANGE_KM"][1] + 1e-9, C["SPACING_STEP_KM"]):
    g = classify(round(s, 4))
    if g is not None: rows.append((round(s, 4), int((g["status"] == "kept").sum())))
SP = pd.DataFrame(rows, columns=["spacing_km", "n_kept"])
cand_sp = SP[(SP.n_kept >= C["N_MIN"]) & (SP.n_kept <= C["N_MAX"])].copy()
if cand_sp.empty: raise RuntimeError(f"No spacing gives {C['N_MIN']}-{C['N_MAX']} sites. Closest: {SP.iloc[(SP.n_kept-C['TARGET_N']).abs().argsort()[:3]].to_dict('records')}")
cand_sp["d"] = (cand_sp.n_kept - C["TARGET_N"]).abs()
best = cand_sp.sort_values(["d", "spacing_km"], ascending=[True, False]).iloc[0]
SPACING_KM = float(best.spacing_km)
print(f"spacing range giving {C['N_MIN']}-{C['N_MAX']} sites: {cand_sp.spacing_km.min():.2f}-{cand_sp.spacing_km.max():.2f} km")
print(f"CHOSEN spacing = {SPACING_KM:.2f} km  ->  {int(best.n_kept)} kept sites")

G = classify(SPACING_KM); st = pd.Series(G["status"])
COUNTS = dict(grid_points_in_padded_bbox=int(len(st)), outside_polygon=int((st == "outside_polygon").sum()),
              inside_polygon=int((st != "outside_polygon").sum()), not_land=int((st == "not_land").sum()),
              land_points=int((st.isin(["nan", "kept"])).sum()), dropped_nan=int((st == "nan").sum()), kept=int((st == "kept").sum()))
print("counts per step:", json.dumps(COUNTS))
# which layer causes the NaN drops (land points only)
lm = st.isin(["nan", "kept"]).values
for n in REQ: print(f"  NaN in {n}: {int(np.isnan(sample(n, G['lon'][lm], G['lat'][lm])).sum())} of {lm.sum()} land points")

**Why bilinear sampling (d):** GWA values are raster cells (0.0025° ≈ 270 m); a site rarely sits on a cell centre. Nearest-neighbour gives a staircase (neighbouring sites can jump by a whole cell); bilinear is continuous, stays within the range of the 4 surrounding cells, and does not invent values. Cost: it slightly smooths ridge tops. **At NaN neighbours** the result is NaN (any NaN among the 4 cells poisons the weighted sum), so such sites are *dropped*, not filled — no invented wind speeds. Pixel values are treated as values at cell centres; I did not verify GWA's exact convention (if cells are area means this is a ≤ half-cell, ~135 m, effect).

In [ ]:
# 6. Build the site table + sample layers (d)
k = G["status"] == "kept"
sites = pd.DataFrame(dict(lon=G["lon"][k], lat=G["lat"][k], x_utm=G["x"][k], y_utm=G["y"][k]))
sites.insert(0, "site_id", [f"W{i+1:03d}" for i in range(len(sites))])
for h in C["HEIGHTS"]: sites[f"ws{h}"] = sample(f"ws{h}", sites.lon.values, sites.lat.values)
for key in ("iec1", "iec2", "iec3"):
    sites[f"cf100_{key}"] = sample(f"cf100_{key}", sites.lon.values, sites.lat.values) if key in CF else np.nan
print(sites.describe().loc[["count", "mean", "min", "max"], ["ws50", "ws100", "ws150"]].round(2))

In [ ]:
# E. Accessibility candidates (e): OpenStreetMap via Overpass (or manual CSV)
S_, W_, N_, E_ = poly_ll.bounds[1], poly_ll.bounds[0], poly_ll.bounds[3], poly_ll.bounds[2]
BB = f"({S_:.5f},{W_:.5f},{N_:.5f},{E_:.5f})"
Q_SUB = f'[out:json][timeout:90];nwr["power"="substation"]{BB};out tags center;'
Q_ROAD = f'[out:json][timeout:120];(way["highway"~"^(motorway|trunk|primary)$"]{BB};node["highway"="motorway_junction"]{BB};);out body;>;out skel qt;'
RANK = {"motorway": 0, "trunk": 1, "primary": 2}

def overpass(q):
    last = None
    for url in C["OVERPASS_URLS"]:
        try:
            r = requests.post(url, data={"data": q}, timeout=150, headers={"User-Agent": "wind-course-project/1.0"})
            r.raise_for_status(); return r.json()
        except Exception as e: last = e; print(f"[!] Overpass failed at {url}: {repr(e)[:120]}")
    raise RuntimeError(f"All Overpass endpoints failed: {last!r}")

def _volt(v):
    try: return max(int(x) for x in re.findall(r"\d+", str(v).replace(";", " ")))
    except ValueError: return 0

def parse_substations(js):
    out = []
    for el in js.get("elements", []):
        la, lo = (el.get("lat"), el.get("lon")) if el["type"] == "node" else (el.get("center", {}).get("lat"), el.get("center", {}).get("lon"))
        if la is None: continue
        t = el.get("tags", {})
        out.append(dict(type="substation", lat=la, lon=lo, name=t.get("name") or t.get("name:en") or "", volt=_volt(t.get("voltage", "")),
                        note=f"osm {el['type']}/{el['id']}; voltage={t.get('voltage','?')}; operator={t.get('operator','?')}"))
    out.sort(key=lambda d: (-d["volt"], d["name"], d["lat"]))
    return out

def parse_junctions(js):
    nodes, ways = {}, []
    for el in js.get("elements", []):
        if el["type"] == "node":
            if el["id"] not in nodes or el.get("tags"): nodes[el["id"]] = el
        elif el["type"] == "way": ways.append(el)
    nw = {}
    for w in ways:
        t = w.get("tags", {}); key = t.get("ref") or t.get("name") or f"way{w['id']}"
        for n in set(w.get("nodes", [])): nw.setdefault(n, []).append((key, t.get("highway", "primary")))
    jn = []
    for nid, lst in nw.items():
        if nid in nodes and len({k for k, _ in lst}) >= 2:            # >=2 distinct roads share the node
            jn.append((min(RANK[h] for _, h in lst if h in RANK), nodes[nid], sorted({k for k, _ in lst})))
    for nid, el in nodes.items():                                       # explicit motorway_junction nodes
        if el.get("tags", {}).get("highway") == "motorway_junction" and nid not in nw:
            jn.append((0, el, [el["tags"].get("ref") or el["tags"].get("name") or "motorway_junction"]))
    jn.sort(key=lambda t: (t[0], t[1]["lon"]))
    # greedy clustering: keep the highest-class node, drop others within JCT_CLUSTER_KM
    kept = []; r2 = (C["JCT_CLUSTER_KM"] * 1000) ** 2
    for rk, el, keys in jn:
        x, y = to_utm.transform(el["lon"], el["lat"])
        for kk in kept:
            if (kk["x"] - x) ** 2 + (kk["y"] - y) ** 2 < r2: kk["roads"] |= set(keys); break
        else: kept.append(dict(rank=rk, lat=el["lat"], lon=el["lon"], x=x, y=y, roads=set(keys)))
    inv = {v: k_ for k_, v in RANK.items()}
    return [dict(type="road", lat=d["lat"], lon=d["lon"], name=inv[d["rank"]] + " junction", volt=0,
                 note="roads: " + ", ".join(sorted(d["roads"]))[:80]) for d in kept]

def read_manual_csv():
    f = C["ACCESS_CSV"]
    if not os.path.exists(f): raise RuntimeError(f"No accessibility candidates: Overpass unavailable/empty and {f} not found. Create a CSV with columns id,type,lat,lon,note (type = road|substation) from OpenStreetMap coordinates you checked yourself.")
    d = pd.read_csv(f)
    if not {"id", "type", "lat", "lon"} <= set(d.columns) or not d.type.isin(["road", "substation"]).all(): raise ValueError("CSV needs id,type(road|substation),lat,lon[,note]")
    d["name"] = d.get("note", ""); d["note"] = d.get("note", ""); return d

cand = None
if C["ACCESS_SOURCE"] == "overpass":
    try:
        subs, jcts = parse_substations(overpass(Q_SUB)), parse_junctions(overpass(Q_ROAD))
        print(f"Overpass: {len(subs)} substations, {len(jcts)} junction clusters in bbox")
        if not subs or not jcts: print("[!] one type came back empty -> add rows via CSV (ACCESS_SOURCE='csv') or choose only from what exists")
        cand = pd.DataFrame([dict(id=f"SUB{i+1:02d}", **d) for i, d in enumerate(subs)] + [dict(id=f"JCT{i+1:02d}", **d) for i, d in enumerate(jcts)])
    except Exception as e: print(f"[!] Overpass route failed ({e}). Falling back to manual CSV.")
if cand is None or cand.empty: cand = read_manual_csv()
cand["inside_polygon"] = contains_xy(poly_ll, cand.lon.values, cand.lat.values)
cand[["x_utm", "y_utm"]] = np.column_stack(to_utm.transform(cand.lon.values, cand.lat.values))
cand.drop(columns=["volt"], errors="ignore").to_csv(OUT / "access_candidates.csv", index=False)
pd.set_option("display.max_rows", 300, "display.max_colwidth", 70, "display.width", 200)
print(cand[["id", "type", "lat", "lon", "inside_polygon", "name", "note"]].round(5).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 7)); ext = [rbox.bounds[0], rbox.bounds[2], rbox.bounds[1], rbox.bounds[3]]
ax.imshow(WS[100], extent=ext, cmap="Greys", alpha=.5); ax.plot(*poly_ll.exterior.xy, "k-", lw=1)
for _, r_ in cand.iterrows():
    sub = r_.type == "substation"
    ax.scatter(r_.lon, r_.lat, marker="*" if sub else "s", s=110 if sub else 50, c="#F0E442" if sub else "#56B4E9", edgecolor="k", zorder=3)
    ax.annotate(r_.id, (r_.lon, r_.lat), xytext=(4, 4), textcoords="offset points", fontsize=7)
ax.set_title("Accessibility candidates (yellow star = substation, blue square = road junction)"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
fig.savefig(OUT / "access_candidates.png", dpi=150, bbox_inches="tight"); plt.show()
print("\n>>> Pick 5-7 IDs (>=2 substations), put them in CONFIG['CHOSEN_IDS'] (cell 2), then run cell E2 and below.")

In [ ]:
# E2. Apply your choice (e/f)
ids = list(CONFIG["CHOSEN_IDS"])   # CONFIG (not C): C still points to the dict from before cell 2 was edited and re-run
if not ids: raise RuntimeError("CONFIG['CHOSEN_IDS'] is empty: choose IDs from the table above, edit cell 2, re-run cell 2 and this cell (re-running E is not needed).")
unk = [i for i in ids if i not in set(cand.id)]
if unk: raise ValueError(f"Unknown IDs: {unk}")
ACC = cand[cand.id.isin(ids)].set_index("id").loc[ids].reset_index()
n_sub = int((ACC.type == "substation").sum())
if not (C["MIN_POINTS"] <= len(ACC) <= C["MAX_POINTS"]) or n_sub < C["MIN_SUBSTATIONS"]:
    raise ValueError(f"Need {C['MIN_POINTS']}-{C['MAX_POINTS']} points with >= {C['MIN_SUBSTATIONS']} substations; got {len(ACC)} points, {n_sub} substations")
# distances in projected CRS (UTM 39N)
d = np.sqrt((sites.x_utm.values[:, None] - ACC.x_utm.values[None, :]) ** 2 + (sites.y_utm.values[:, None] - ACC.y_utm.values[None, :]) ** 2) / 1000
sites["dist_km"] = d.min(axis=1); sites["nearest_access_id"] = ACC.id.values[d.argmin(axis=1)]
sites["access_score"] = np.exp(-C["ACCESS_DECAY_PER_KM"] * sites.dist_km)
print(ACC[["id", "type", "lat", "lon"]].to_string(index=False)); print(sites[["dist_km", "access_score"]].describe().round(3))

In [ ]:
# 7. Sanity checks (h)
res = []
def chk(name, ok, info=""): res.append(ok); print(f"{'PASS' if ok is True else ('SKIP' if ok is None else 'FAIL')}  {name}  {info}")
chk("200-300 rows", C["N_MIN"] <= len(sites) <= C["N_MAX"], f"(n={len(sites)})")
reqcols = ["site_id", "lon", "lat", "x_utm", "y_utm"] + REQ + ["dist_km", "nearest_access_id", "access_score"]
chk("no NaN in required columns", not sites[reqcols].isna().any().any())
sh = float(((sites.ws50 < sites.ws100) & (sites.ws100 < sites.ws150)).mean())
chk("ws50<ws100<ws150 for >=95% of sites", sh >= 0.95, f"(share = {sh*100:.2f}%)")
chk("speeds plausible (all between 3 and 20 m/s)", bool(((sites[["ws50", "ws100", "ws150"]] >= 3) & (sites[["ws50", "ws100", "ws150"]] <= 20)).all().all()), f"(range {sites[['ws50','ws100','ws150']].min().min():.2f}-{sites[['ws50','ws100','ws150']].max().max():.2f})")
chk("all sites inside polygon", bool(contains_xy(poly_utm, sites.x_utm.values, sites.y_utm.values).all()))
chk("all sites on land (unbuffered NE land - lakes)", None if C["LAND_MODE"] == "polygon_only" else bool(contains_xy(LAND_UTM, sites.x_utm.values, sites.y_utm.values).all()))
chk("distances >= 0", bool((sites.dist_km >= 0).all()))
chk("access_score in (0,1]", bool(((sites.access_score > 0) & (sites.access_score <= 1)).all()))
chk("unique site_id", sites.site_id.is_unique)
print("OVERALL:", "ALL PASS" if all(r is not False for r in res) else "SOME FAIL")

In [ ]:
# 8. Save outputs (g)
cols = ["site_id", "lon", "lat", "x_utm", "y_utm", "ws50", "ws100", "ws150", "cf100_iec1", "cf100_iec2", "cf100_iec3", "dist_km", "nearest_access_id", "access_score"]
sites[cols].round({"lon": 6, "lat": 6, "x_utm": 1, "y_utm": 1, "ws50": 3, "ws100": 3, "ws150": 3, "dist_km": 4, "access_score": 6}).to_csv(OUT / "candidate_sites.csv", index=False)
ACC[["id", "type", "lat", "lon", "x_utm", "y_utm", "name", "note"]].round({"lat": 6, "lon": 6, "x_utm": 1, "y_utm": 1}).to_csv(OUT / "accessibility_points.csv", index=False)
nan150 = int(np.isnan(WS[150]).sum()); nan150_in = int(np.isnan(WS[150][inside]).sum())
cfg = dict(polygon_file=str(C["POLYGON_PATH"]), polygon_area_km2=round(poly_utm.area / 1e6, 1), polygon_bounds_lonlat=list(poly_ll.bounds),
           grid_bbox_lonlat_padded=list(BBOX_LL), bbox_pad_deg=C["BBOX_PAD_DEG"], crs_data="EPSG:4326", crs_projected=C["UTM"],
           grid_spacing_km=SPACING_KM, spacing_search=dict(target_n=C["TARGET_N"], allowed=[C["N_MIN"], C["N_MAX"]], step_km=C["SPACING_STEP_KM"]),
           heights_m=C["HEIGHTS"], files_used={**{f"ws{h}": p for h, p in WS_PATHS.items()}, **{f"cf100_{k}": p for k, p in CF_PATHS.items()}},
           cf_available=sorted(CF), cf_required=C["CF_REQUIRED"], land_mask_mode=C["LAND_MODE"], mask_source=MASK_SOURCE, shore_buffer_m=C["SHORE_BUFFER_M"],
           sampling_method="bilinear (scipy RegularGridInterpolator, pixel centres); any NaN neighbour -> NaN -> site dropped",
           counts=COUNTS, ws150_nan_pixels_total=nan150, ws150_nan_pixels_in_polygon=nan150_in,
           access=dict(chosen_ids=ids, n_substations=n_sub, score="exp(-%.2f*d_km)" % C["ACCESS_DECAY_PER_KM"], source=C["ACCESS_SOURCE"]),
           seed=C["SEED"], ordering_share_ws50_100_150=round(sh, 4),
           notes=["92 m rotor on a 50 m tower leaves only about 4 m ground clearance (limitation for the report)"])
(OUT / "region_config.json").write_text(json.dumps(cfg, indent=2)); print(json.dumps(cfg["counts"])); print(os.listdir(OUT))

In [ ]:
# 9. Plots (i)
COL = dict(kept="white", not_land="#CC79A7", nan="#E69F00")
polyx, polyy = poly_ll.exterior.xy
ext = [rbox.bounds[0], rbox.bounds[2], rbox.bounds[1], rbox.bounds[3]]
def base(ax, arr, title, cmap="viridis"):
    ax.imshow(arr, extent=ext, cmap=cmap, alpha=.3); ax.imshow(np.where(inside, arr, np.nan), extent=ext, cmap=cmap)
    ax.plot(polyx, polyy, "k-", lw=1.2); ax.set_title(title); ax.set_xlabel("lon"); ax.set_ylabel("lat")
st_ = G["status"]
# (1) ws100 map
fig, ax = plt.subplots(figsize=(10, 8)); base(ax, WS[100], "ws100 (m/s), sites and removed grid points")
for s_, mk, lab in [("not_land", "x", "removed: not land"), ("nan", "^", "removed: NaN layer"), ("kept", "o", "kept site")]:
    m = st_ == s_; ax.scatter(G["lon"][m], G["lat"][m], marker=mk, s=16 if s_ == "kept" else 26, c=COL[s_], edgecolor="k" if s_ == "kept" else None, lw=.5, label=f"{lab} ({m.sum()})", zorder=3)
for _, r_ in ACC.iterrows():
    sub = r_.type == "substation"; ax.scatter(r_.lon, r_.lat, marker="*" if sub else "s", s=220 if sub else 90, c="#F0E442" if sub else "#56B4E9", edgecolor="k", zorder=4, label=f"{r_.type}: {r_.id}")
ax.legend(fontsize=7, loc="lower left"); fig.savefig(OUT / "map_ws100_sites.png", dpi=150, bbox_inches="tight"); plt.show()
# (2) histogram
fig, ax = plt.subplots(figsize=(6, 4)); ax.hist(sites.ws100, bins=25, color="#0072B2", edgecolor="w"); ax.axvline(sites.ws100.mean(), c="k", ls="--", label=f"mean {sites.ws100.mean():.2f}")
ax.set_xlabel("ws100 at sites (m/s)"); ax.set_ylabel("sites"); ax.legend(); fig.savefig(OUT / "hist_ws100.png", dpi=150, bbox_inches="tight"); plt.show()
# (3) ws150 NaN zone
fig, ax = plt.subplots(figsize=(7, 6)); nanz = np.where(np.isnan(WS[150]) & inside, 1.0, np.where(inside, 0.0, np.nan))
ax.imshow(nanz, extent=ext, cmap="Reds", vmin=0, vmax=1.6); ax.plot(polyx, polyy, "k-", lw=1.2)
for s_, mk in [("kept", "o"), ("nan", "^")]:
    m = st_ == s_; ax.scatter(G["lon"][m], G["lat"][m], marker=mk, s=14, c=COL[s_], edgecolor="k", lw=.4, label=f"{s_} ({m.sum()})")
ax.set_title(f"ws150 NaN zone inside polygon (red, {nan150_in} px) vs sites"); ax.legend(fontsize=8); ax.set_xlabel("lon"); ax.set_ylabel("lat")
fig.savefig(OUT / "map_ws150_nan_zone.png", dpi=150, bbox_inches="tight"); plt.show()

**Limitations to carry into the report:** (1) a 92 m rotor on a 50 m tower leaves only about 4 m ground clearance; (2) the land mask is 1:10m and the Caspian level changes; (3) sites in the ws150 NaN zone are dropped, which can bias the site set away from the polygon's west edge; (4) accessibility points are your own choice from OSM, distance is straight-line, not road length.